# 9jaLingo vLLM TTS — GPU Test (Colab)

This notebook tests the 9jaLingo vLLM TTS server on a Colab GPU.

**Requirements:** Select **GPU runtime** (Runtime → Change runtime type → T4 GPU)

## Setup

In [ ]:
# Check GPU is available
!nvidia-smi

In [ ]:
# Clone the repo (replace with your fork if private)
!git clone https://github.com/9jaLingo/9jalingo-vllm.git
%cd 9jalingo-vllm

In [ ]:
# Set HuggingFace token for private model access
# Option 1: Use Colab secrets (recommended)
try:
    from google.colab import userdata
    import os
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab secrets')
except Exception:
    print('⚠️  Could not load from Colab secrets.')
    print('Set it manually below:')

# Option 2: Set manually (uncomment and paste your token)
# import os
# os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

In [ ]:
# Install dependencies (GPU mode)
# 1. Install the project + its deps (naijalingo-tts-2, fastapi, etc.)
!pip install -e . --quiet

# 2. Install vLLM (GPU version)
!pip install vllm --quiet

print('\n✅ All dependencies installed')

In [ ]:
# Verify GPU is visible to PyTorch and vLLM
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import vllm
print(f'vLLM: {vllm.__version__}')

## Prepare Model

Download the HF model and strip custom weights incompatible with vLLM's `Lfm2ForCausalLM`.

In [ ]:
from prepare_model import prepare

model_path = prepare()
print(f'\n✅ vLLM-compatible model ready at: {model_path}')

## Start the Server

Launch the FastAPI server in the background, then test it with HTTP requests.

In [ ]:
# Start the server in background
import subprocess, time, os

env = os.environ.copy()
server_proc = subprocess.Popen(
    ['python', 'server.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for server to be ready
print('Starting server... (this takes ~1-2 min for model loading + warmup)')
import urllib.request
for i in range(180):  # wait up to 3 min
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:8000/health')
        print(f'\n✅ Server is ready! (took ~{(i+1)*2}s)')
        break
    except Exception:
        print('.', end='', flush=True)
else:
    print('\n❌ Server did not start in time. Check logs below:')
    server_proc.terminate()
    out, _ = server_proc.communicate(timeout=5)
    print(out[-3000:] if out else 'No output')

In [ ]:
# Quick health check + list available speakers
import requests

# Health
r = requests.get('http://localhost:8000/health')
print('Health:', r.json())

# Speakers
r = requests.get('http://localhost:8000/v1/speakers')
speakers = r.json()
print(f"\nSpeakers loaded: {speakers['total']}")
print(f"By language: {speakers['by_language']}")
print(f"\nSample speakers: {speakers['speakers'][:5]}")

## Test 1: Basic TTS (Language Tag Only)

Uses the vLLM fast path — no speaker embedding, just language tag routing.

In [ ]:
import requests, soundfile as sf, io, os
from IPython.display import Audio, display

OUTPUT_DIR = "tts_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def tts_request(text, language='pcm', speaker=None, save_as=None, **kwargs):
    """Make a TTS request, return audio data + sample rate, optionally save to file.

    Server defaults: temperature=0.6, top_p=0.85, repetition_penalty=1.3
    (matches the working naijalingo-tts-2 inference notebook).
    """
    payload = {
        'text': text,
        'model': '9javox',
        'language': language,
        'response_format': 'wav',
    }
    # Only override generation params if explicitly passed
    for key in ('temperature', 'top_p', 'repetition_penalty', 'max_chunk_duration', 'silence_duration'):
        if key in kwargs:
            payload[key] = kwargs[key]
    if speaker:
        payload['speaker'] = speaker

    resp = requests.post('http://localhost:8000/v1/audio/speech', json=payload)
    if resp.status_code != 200:
        print(f'Error {resp.status_code}: {resp.text[:500]}')
        resp.raise_for_status()
    data, sr = sf.read(io.BytesIO(resp.content))

    if save_as:
        out_path = os.path.join(OUTPUT_DIR, save_as)
        sf.write(out_path, data, sr)
        print(f'Saved: {out_path}')

    return data, sr

# Nigerian Pidgin
text_pcm = "My people, if patience were food, Nigerians would never go hungry!"
data, sr = tts_request(text_pcm, language='pcm', save_as='pidgin_test.wav')
print(f'Pidgin: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Yoruba
text_yo = "Ẹ kú ilé o, a dúpẹ́ fún ọjọ́ tuntun yìí."
data, sr = tts_request(text_yo, language='yo', save_as='yoruba_test.wav')
print(f'Yoruba: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Hausa
text_ha = "Sannu da zuwa, muna farin ciki da ganin ku."
data, sr = tts_request(text_ha, language='ha', save_as='hausa_test.wav')
print(f'Hausa: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Igbo
text_ig = "Nnọọ, anyị nwere obi ụtọ ịhụ unu."
data, sr = tts_request(text_ig, language='ig', save_as='igbo_test.wav')
print(f'Igbo: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 2: Speaker Embedding Path

Uses the Direct model path with a pre-computed speaker embedding.
This loads a second model copy — requires ~3GB+ VRAM on top of the vLLM engine.

In [ ]:
# Test with a specific speaker (uses Direct model path)
text = "Wetin dey happen, how body na?"
data, sr = tts_request(text, language='pcm', speaker='ada_pcm', save_as='speaker_ada_pcm.wav')
print(f'Speaker ada_pcm: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Another speaker
text = "Ẹ̀yin ọmọ Yorùbá, ẹ kú iṣẹ́ o!"
data, sr = tts_request(text, language='yo', speaker='adeola_yo', save_as='speaker_adeola_yo.wav')
print(f'Speaker adeola_yo: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 3: Long-form Generation

Tests sentence chunking + silence insertion for longer text.

In [ ]:
long_text = (
    "My people, if patience were food, Nigerians would never go hungry! "
    "But since we cannot cook patience for dinner, let us cook progress instead "
    "and season it with hard work, unity, and small small enjoyment. "
    "As we dey go, make we no forget say the journey wey start with one step, "
    "fit carry us to where we never imagine before."
)
print(f'Text length: {len(long_text)} chars, estimated: {len(long_text)/15:.1f}s')

data, sr = tts_request(long_text, language='pcm', save_as='long_form_test.wav')
print(f'Long-form (vLLM): {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# Test long-form with speaker embedding (Direct model path)
data, sr = tts_request(long_text, language='pcm', speaker='ada_pcm', save_as='long_form_speaker_test.wav')
print(f'Long-form (Direct/speaker): {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 4: SSE Streaming

Tests the Server-Sent Events streaming path — audio chunks arrive incrementally
as the model generates, instead of waiting for the full utterance.

In [ ]:
import requests, json, base64, numpy as np, time
from IPython.display import Audio, display

def tts_stream(text, language='pcm', speaker=None):
    """SSE streaming TTS request — returns (audio_np, sample_rate)."""
    payload = {
        'text': text,
        'model': '9javox',
        'language': language,
        'stream_format': 'sse',
    }
    if speaker:
        payload['speaker'] = speaker

    resp = requests.post(
        'http://localhost:8000/v1/audio/speech',
        json=payload,
        stream=True,
    )
    resp.raise_for_status()

    pcm_chunks = []
    meta = {}
    first_chunk_time = None
    start = time.time()

    for line in resp.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data: '):
            continue
        event = json.loads(line[6:])

        if event['type'] == 'speech.audio.delta':
            pcm_bytes = base64.b64decode(event['audio'])
            pcm_chunks.append(np.frombuffer(pcm_bytes, dtype=np.int16))
            if first_chunk_time is None:
                first_chunk_time = time.time() - start

        elif event['type'] == 'speech.audio.done':
            meta = event.get('usage', {})

        elif event['type'] == 'error':
            raise RuntimeError(f"SSE error: {event['error']}")

    total_time = time.time() - start

    if not pcm_chunks:
        raise RuntimeError('No audio received from SSE stream')

    audio_int16 = np.concatenate(pcm_chunks)
    audio_float = audio_int16.astype(np.float32) / 32767.0
    sr = 22050

    print(f'  Time to first chunk: {first_chunk_time:.2f}s')
    print(f'  Total stream time:   {total_time:.2f}s')
    print(f'  Audio duration:      {len(audio_float)/sr:.1f}s')
    if meta:
        print(f'  Usage:               {meta}')

    return audio_float, sr

# vLLM SSE streaming (short text)
print('--- vLLM SSE Streaming ---')
data, sr = tts_stream("How body na, everything dey kampe?", language='pcm')
display(Audio(data, rate=sr))

In [ ]:
# Direct SSE streaming (with speaker embedding)
print('--- Direct SSE Streaming (speaker=ada_pcm) ---')
data, sr = tts_stream("Wetin dey happen for this side?", language='pcm', speaker='ada_pcm')
display(Audio(data, rate=sr))

## Test 5: Voice Cloning

Two-step voice cloning:
1. Upload a reference audio to `/v1/voice/clone` → get a 128-dim speaker embedding
2. Pass that embedding to `/v1/audio/speech` via `speaker_embedding`

Also tests the one-shot `/v1/voice/clone/generate` endpoint.

In [ ]:
# Step 1: Generate a reference audio to use as the cloning source
# (In production you'd upload a real voice recording)
ref_text = "My name is Ada, I come from Lagos and I dey happy to meet you."
ref_data, ref_sr = tts_request(ref_text, language='pcm', speaker='ada_pcm', save_as='clone_reference.wav')
print(f'Reference audio: {len(ref_data)/ref_sr:.1f}s')
display(Audio(ref_data, rate=ref_sr))

# Step 2: Extract speaker embedding via /v1/voice/clone
ref_path = os.path.join(OUTPUT_DIR, 'clone_reference.wav')
with open(ref_path, 'rb') as f:
    resp = requests.post(
        'http://localhost:8000/v1/voice/clone',
        files={'file': ('clone_reference.wav', f, 'audio/wav')},
    )
resp.raise_for_status()
clone_result = resp.json()
print(f"\nExtracted embedding: dim={clone_result['dim']}")
print(f"First 5 values: {clone_result['embedding'][:5]}")

# Step 3: Generate speech using the cloned embedding
clone_text = "I don tell you say na Lagos life be the best life, nobody fit change my mind!"
payload = {
    'text': clone_text,
    'model': '9javox',
    'language': 'pcm',
    'response_format': 'wav',
    'speaker_embedding': clone_result['embedding'],
}
resp = requests.post('http://localhost:8000/v1/audio/speech', json=payload)
resp.raise_for_status()
data, sr = sf.read(io.BytesIO(resp.content))
sf.write(os.path.join(OUTPUT_DIR, 'cloned_voice.wav'), data, sr)
print(f'\nCloned voice output: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

In [ ]:
# One-shot clone + generate via /v1/voice/clone/generate
ref_path = os.path.join(OUTPUT_DIR, 'clone_reference.wav')
with open(ref_path, 'rb') as f:
    resp = requests.post(
        'http://localhost:8000/v1/voice/clone/generate',
        files={'file': ('clone_reference.wav', f, 'audio/wav')},
        data={
            'text': 'Omo, this Lagos traffic no go ever change, but we dey move!',
            'language': 'pcm',
            'response_format': 'wav',
        },
    )
resp.raise_for_status()
data, sr = sf.read(io.BytesIO(resp.content))
sf.write(os.path.join(OUTPUT_DIR, 'clone_oneshot.wav'), data, sr)
print(f'One-shot clone + generate: {len(data)/sr:.1f}s @ {sr}Hz')
display(Audio(data, rate=sr))

## Test 6: Performance Benchmark (RTF)

Measure Real-Time Factor: how fast the GPU generates compared to audio duration.

In [ ]:
import time

texts = {
    'pcm': "How far, everything dey alright?",
    'yo': "Bawo ni, ṣe daadaa ni?",
    'ha': "Ina kwana, lafiya lau?",
    'ig': "Kedụ, ọ dị mma?",
}

print(f'{"Lang":<6} {"Gen Time":<12} {"Audio Dur":<12} {"RTF":<8}')
print('-' * 40)

for lang, text in texts.items():
    start = time.time()
    data, sr = tts_request(text, language=lang)
    gen_time = time.time() - start
    audio_dur = len(data) / sr
    rtf = gen_time / audio_dur if audio_dur > 0 else 0
    print(f'{lang:<6} {gen_time:<12.2f} {audio_dur:<12.2f} {rtf:<8.2f}')

print('\nRTF < 1.0 = faster than real-time ✅')

## Cleanup

In [ ]:
# Stop the server
server_proc.terminate()
server_proc.wait(timeout=10)
print('✅ Server stopped')